# 🧠 Алгоритми та структури даних · Урок 2 — Базові структури даних

Структура даних — це **спосіб організації даних у пам'яті**, який визначає, які операції дешеві, а
які дорогі. Немає «найкращої» структури — є **правильна під задачу**. Розберемо базові цеглинки:
масиви, зв'язані списки, стек, чергу, дек і хеш-таблиці.

**Передумова:** [Урок 1 — складність](lektsiya-1-skladnist-ta-rekursiya.ipynb).

### Що ви вмітимете
- пояснити різницю **масив vs зв'язаний список** і їхні складності;
- реалізувати **стек**, **чергу**, **зв'язаний список** з нуля;
- розуміти, як влаштована **хеш-таблиця** (колізії, chaining/open addressing) — основа `dict`/`set`.

## 1. Масиви

**Масив** — послідовність елементів у **суцільному** блоці пам'яті. Тому доступ за індексом —
**O(1)** (адреса = початок + i·розмір_елемента).

- **Статичний масив** — фіксований розмір (як у C, Java). У Python «чистий» масив — модуль `array`
  або `numpy`.
- **Динамічний масив** — росте автоматично. Саме такий **Python `list`** (див. амортизацію в Уроці 1).

**Складність масиву/списку:**

| Операція | Складність | Чому |
|----------|-----------|------|
| доступ `arr[i]` | **O(1)** | пряма адресація |
| зміна `arr[i] = x` | **O(1)** | те саме |
| `append` (у кінець) | **O(1)** амортизовано | зрідка перевиділення |
| `insert`/`pop` на початку | **O(n)** | треба зсунути всі елементи |
| пошук значення | **O(n)** | перебір |

> 🎯 **Питання: чому `list.insert(0, x)` повільний?** Бо всі наявні елементи треба **зсунути**
> праворуч на одну позицію — O(n). Якщо часто вставляєте/видаляєте з початку — беріть `deque`.

## 2. Зв'язані списки (linked list)

**Зв'язаний список** — це ланцюг **вузлів** (`Node`), кожен зберігає **значення** і **посилання**
на наступний вузол. Елементи лежать у пам'яті **не поруч**.

```
[10|•]──▶[20|•]──▶[30|•]──▶ None
 head
```

- **Однобічний (singly)** — вузол знає лише `next`.
- **Двобічний (doubly)** — вузол знає `next` і `prev` (можна ходити в обидва боки).

Плюс: вставка/видалення **на відомій позиції** — O(1) (перечепити посилання, без зсуву).
Мінус: **немає** доступу за індексом — щоб дійти до i-го, треба пройти від голови → **O(n)**.

In [ ]:
# Реалізація однобічного зв'язаного списку
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None          # посилання на наступний вузол

class LinkedList:
    def __init__(self):
        self.head = None

    def push_front(self, value):          # вставка на початок -> O(1)
        node = Node(value)
        node.next = self.head
        self.head = node

    def append(self, value):              # вставка в кінець -> O(n) (треба дійти до кінця)
        node = Node(value)
        if self.head is None:
            self.head = node
            return
        cur = self.head
        while cur.next is not None:
            cur = cur.next
        cur.next = node

    def find(self, value):                # пошук -> O(n)
        cur = self.head
        while cur is not None:
            if cur.value == value:
                return True
            cur = cur.next
        return False

    def to_list(self):
        out, cur = [], self.head
        while cur is not None:
            out.append(cur.value)
            cur = cur.next
        return out

ll = LinkedList()
ll.append(10); ll.append(20); ll.append(30)
ll.push_front(5)
print("список:", ll.to_list())     # [5, 10, 20, 30]
print("є 20? ", ll.find(20))       # True
print("є 99? ", ll.find(99))       # False

### Масив vs зв'язаний список — коли що

| Критерій | Масив / `list` | Зв'язаний список |
|----------|----------------|------------------|
| доступ за індексом | **O(1)** ✅ | O(n) ❌ |
| вставка/видалення на початку | O(n) ❌ | **O(1)** ✅ |
| вставка в кінець | O(1) аморт. | O(1)* (якщо є `tail`) |
| пам'ять | компактно | +посилання на кожен вузол |
| кеш процесора | дружній (поруч) | недружній (розкидано) |

> 🔎 **Важливо знати.** На практиці масив (`list`) перемагає майже завжди — через доступ за
> індексом і дружність до кеша. Зв'язані списки корисні там, де багато вставок/видалень посередині
> **без** потреби в індексації, і як основа інших структур (стек, черга, adjacency list).

## 3. Стек (stack) — LIFO

**Стек** працює за принципом **LIFO** (Last In, First Out) — «останнім прийшов, першим пішов», як
стопка тарілок. Дві головні операції: **push** (покласти зверху) і **pop** (зняти зверху), обидві **O(1)**.

У Python стек — це просто `list` (`append` / `pop`).

**Де застосовується:** скасування дій (Ctrl+Z), стек викликів функцій, обхід DFS, перевірка
збалансованості дужок, обчислення виразів.

In [ ]:
# Застосування стека: перевірка збалансованості дужок ()[]{}
def is_balanced(s):
    stack = []
    pairs = {")": "(", "]": "[", "}": "{"}
    for ch in s:
        if ch in "([{":
            stack.append(ch)              # push відкриваючу
        elif ch in ")]}":
            if not stack or stack.pop() != pairs[ch]:   # pop і звіряємо
                return False
    return not stack                      # стек має лишитись порожнім

for expr in ["(a[b]{c})", "([)]", "(((", "{[()]}"]:
    print(f"{expr:>10} -> {is_balanced(expr)}")

## 4. Черга (queue) — FIFO

**Черга** працює за принципом **FIFO** (First In, First Out) — «першим прийшов, першим пішов», як
черга в магазині. Операції: **enqueue** (додати в кінець) і **dequeue** (взяти з початку).

⚠️ **Не** використовуйте `list.pop(0)` для черги — це **O(n)** (зсув усіх елементів)! Правильно —
`collections.deque`, де обидва кінці **O(1)**.

**Де застосовується:** обхід BFS, черги задач, буфери, планувальники.

In [ ]:
from collections import deque

queue = deque()
queue.append("A")        # enqueue -> O(1)
queue.append("B")
queue.append("C")
print("черга:", list(queue))
print("обслуговуємо:", queue.popleft())   # dequeue з початку -> O(1)  -> A
print("обслуговуємо:", queue.popleft())   # B
print("лишилось:", list(queue))           # [C]

## 5. Дек (deque) — двобічна черга

**Дек** (double-ended queue) — черга, у якій можна додавати/видаляти з **обох** кінців за O(1).
`collections.deque` — універсальний інструмент: працює і як стек, і як черга.

| Операція | `deque` |
|----------|---------|
| `append` / `pop` (правий кінець) | O(1) |
| `appendleft` / `popleft` (лівий) | O(1) |
| доступ за індексом посередині | O(n) |

In [ ]:
from collections import deque

d = deque([1, 2, 3])
d.appendleft(0)      # 0 1 2 3
d.append(4)          # 0 1 2 3 4
print("дек:", list(d))
print("зліва:", d.popleft(), "| справа:", d.pop())   # 0 | 4
print("лишилось:", list(d))                           # [1, 2, 3]

## 6. 🔎 Хеш-таблиці (hash table)

**Хеш-таблиця** дає доступ/вставку/пошук за ключем у **середньому O(1)** — і саме на ній
побудовані Python `dict` та `set`. Ідея: перетворити ключ на число (**хеш**) і за ним обчислити
**індекс комірки** в масиві.

```
ключ  --hash()--> велике число --% розмір--> індекс комірки в масиві
"cat" --------->  3900284...   --------->     3
```

### Колізії
Різні ключі можуть дати **однаковий** індекс — це **колізія**. Два способи розв'язання:

1. **Ланцюжки (chaining)** — у кожній комірці зберігається **список** пар. Колізія → додаємо в список.
2. **Відкрита адресація (open addressing)** — шукаємо **наступну вільну** комірку (linear/quadratic
   probing). Саме цей підхід використовує **CPython** для `dict`/`set`.

```
chaining:                    open addressing:
[0] -> None                  [0] порожньо
[1] -> ("cat",1)->("dog",2)  [1] ("cat",1)
[2] -> None                  [2] ("dog",2)   <- зайняте "cat", поклали далі
[3] -> ("fox",3)             [3] ("fox",3)
```

Реалізуймо **просту** хеш-таблицю з ланцюжками, щоб побачити механізм зсередини.

In [ ]:
# Навчальна хеш-таблиця з ланцюжками (chaining)
class HashMap:
    def __init__(self, capacity=8):
        self.capacity = capacity
        self.buckets = [[] for _ in range(capacity)]   # кожна комірка = список пар (k, v)

    def _index(self, key):
        return hash(key) % self.capacity                # хеш -> індекс комірки

    def put(self, key, value):
        bucket = self.buckets[self._index(key)]
        for i, (k, _) in enumerate(bucket):
            if k == key:                                # ключ уже є -> оновлюємо
                bucket[i] = (key, value)
                return
        bucket.append((key, value))                     # інакше додаємо

    def get(self, key, default=None):
        bucket = self.buckets[self._index(key)]
        for k, v in bucket:
            if k == key:
                return v
        return default

m = HashMap(capacity=4)
m.put("cat", 1); m.put("dog", 2); m.put("fox", 3); m.put("cat", 99)  # cat оновиться
print("cat =", m.get("cat"))          # 99
print("dog =", m.get("dog"))          # 2
print("bird =", m.get("bird", "немає"))
print("розподіл по комірках:", [len(b) for b in m.buckets])   # видно колізії (len>1)

### Складність хеш-таблиці

| Операція | Середній | Найгірший |
|----------|----------|-----------|
| вставка / пошук / видалення | **O(1)** | O(n) (усі ключі в одну комірку) |

Найгірший O(n) трапляється при поганій хеш-функції (усі колізії в один ланцюжок). Тому реальні
реалізації (як CPython) стежать за **коефіцієнтом заповнення** (load factor) і **ресайзять** таблицю,
коли вона заповнюється, тримаючи колізії рідкісними.

> 🎯 **Питання: чому ключі `dict`/`set` мають бути незмінними (hashable)?** Бо позиція елемента
> залежить від його **хешу**. Якби ключ можна було змінити після вставки — його хеш змінився б, і
> елемент «загубився» б у неправильній комірці. Тому ключами можуть бути `str`, `int`, `tuple`, але
> **не** `list`/`dict`. (Згадайте `__hash__` з Модуля 2 ООП.)

# ✅ Підсумок уроку
- **Масив/`list`** — суцільна пам'ять: доступ за індексом O(1), але вставка на початку O(n).
- **Зв'язаний список** — вузли з посиланнями: вставка/видалення O(1), але немає індексації (O(n)).
- **Стек (LIFO)** — `push`/`pop` з одного кінця, O(1); дужки, DFS, undo.
- **Черга (FIFO)** — `deque` (не `list.pop(0)`!); BFS, буфери.
- **Дек** — обидва кінці O(1); універсальний (і стек, і черга).
- **Хеш-таблиця** — хеш → індекс; колізії розв'язують chaining / open addressing; середнє O(1) — основа `dict`/`set`.

### ▶️ Далі
Урок 3 — **дерева, купи та Union-Find**.

### 📚 Хочу знати більше
- Як влаштований Python `dict` (CPython): <https://docs.python.org/3/faq/design.html#how-are-dictionaries-implemented-in-cpython>
- `collections` (deque та ін.): <https://docs.python.org/3/library/collections.html>
- Візуалізація хеш-таблиць: <https://visualgo.net/en/hashtable>